# 03 — Ghép Retrieval và MRC của Huy

Chạy câu hỏi → tìm đoạn văn → trả lời dạng chuỗi, không cần STT, TTS hay web. Demo dùng dữ liệu tự tạo và baseline chọn câu; **không phải ViT5 đã huấn luyện**.

Thứ tự: thiết lập, khởi tạo, hỏi đáp, kiểm tra hợp đồng, nạp checkpoint thật.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/qa_pipeline.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Mở notebook từ trong thư mục vimmrc-qa-voice")
sys.path.insert(0, str(ROOT / "scripts"))
from qa_pipeline import QAPipeline, Retriever, LexicalBaselineClient, MRCClient

## 1. Hai module độc lập
Retriever nhận kho đoạn văn. Reader nhận câu hỏi và đoạn văn được chọn.

In [ ]:
passages = [
    {"id": "capital", "passage": "Hà Nội là thủ đô của Việt Nam. Thành phố nằm ở miền Bắc."},
    {"id": "coffee", "passage": "Cà phê được trồng nhiều ở Tây Nguyên."},
]
pipeline = QAPipeline(Retriever(passages), LexicalBaselineClient())

## 2. Hỏi đáp và xem nguồn
Điểm BM25 là điểm xếp hạng, không phải xác suất đúng.

In [ ]:
result = pipeline.predict("Thủ đô của Việt Nam là gì?")
result

{'question': 'Thủ đô của Việt Nam là gì?',
 'answer': 'Hà Nội là thủ đô của Việt Nam.',
 'passage_id': 'capital',
 'passage': 'Hà Nội là thủ đô của Việt Nam. Thành phố nằm ở miền Bắc.',
 'retrieval_score': 3.7042683333567927}

## 3. Hợp đồng ghép nối
Web hoặc STT truyền `str` vào; TTS nhận `str` từ `answer_question`. Không có từ khớp thì trả thông báo không tìm thấy đoạn văn.

In [ ]:
answer = pipeline.answer_question("Cà phê trồng ở đâu?")
assert isinstance(answer, str) and answer
assert pipeline.predict("zzzzqqqq")["passage_id"] is None
print(answer)

Cà phê được trồng nhiều ở Tây Nguyên.


## 4. Nạp mô hình thật
Huấn luyện và lưu checkpoint bằng notebook MRC; tạo index bằng CLI Retrieval. Sửa đường dẫn rồi bật cờ. Mô hình sinh đáp án được huấn luyện từ dữ liệu trắc nghiệm; cần đánh giá riêng theo mục tiêu sử dụng.

In [ ]:
USE_TRAINED_MODEL = False
MODEL_PATH = ROOT / "mrc-model_Huy/checkpoints/vit5-vimmrc"
INDEX_PATH = ROOT / "retrieval-model_Huy/artifacts/dev_index.json"
if USE_TRAINED_MODEL:
    pipeline = QAPipeline(Retriever.load(INDEX_PATH), MRCClient(str(MODEL_PATH)))
    print(pipeline.answer_question("Nhân vật chính trong bài đọc là ai?"))
else:
    print("Chưa nạp checkpoint; đang dùng lexical baseline.")

Chưa nạp checkpoint; đang dùng lexical baseline.


## 5. Bài tập kiểm chứng
Thêm một đoạn do bạn viết và đặt câu hỏi. Kiểm tra `passage_id` trước khi đánh giá đáp án. Đo riêng Retrieval, MRC với đoạn đúng, và pipeline với đoạn truy xuất; không báo kết quả MRC với đoạn đúng thành kết quả toàn hệ thống.

In [ ]:
extra = {"id": "exercise", "passage": "Đà Lạt thuộc tỉnh Lâm Đồng."}
exercise_pipeline = QAPipeline(Retriever(passages + [extra]), LexicalBaselineClient())
exercise_pipeline.predict("Đà Lạt thuộc tỉnh nào?")

{'question': 'Đà Lạt thuộc tỉnh nào?',
 'answer': 'Đà Lạt thuộc tỉnh Lâm Đồng.',
 'passage_id': 'exercise',
 'passage': 'Đà Lạt thuộc tỉnh Lâm Đồng.',
 'retrieval_score': 4.674590482438865}